# DataSage 1/4 — Cleaning Inference

Loads the **cleaning** LoRA adapter and runs episodes against the live environment.

Also runs the base model (no LoRA) on the same seeds for comparison.

**Requires:** GPU runtime (T4+)  
**Output:** `cleaning_results.json`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/demo/inference/01_cleaning_inference.ipynb)

In [ ]:
!pip install -q unsloth trl requests datasets huggingface_hub

In [ ]:
import os, json, re, time, requests
import numpy as np
import torch
from datetime import datetime

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# === Config ===
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"
LORA_REPO = "ricalanis/cleaning-grpo"
ENV_URL = "https://ricalanis-datasage-cleaning.hf.space"
MAX_SEQ_LENGTH = 2048

DOMAINS = ["hr", "sales", "pm", "it_ops"]
EPISODES_PER_DOMAIN = 3
MAX_STEPS = 15

SYSTEM_PROMPT = (
    "You are a data quality agent. You clean enterprise datasets across multiple "
    "domains (HR, Sales, Project Management, IT Operations).\n\n"
    "Each turn, analyze the data and respond with a JSON cleaning action:\n"
    '{"operation": "<op>", "column": "<col>", "value": "<val>", "params": {}}\n\n'
    "Available operations:\n"
    "- fill_null: Fill null values (value can be 'median', 'mode', or a specific value)\n"
    "- fix_type: Fix type mismatches in a column (cast to proper type)\n"
    "- remove_duplicate: Remove duplicate rows\n"
    "- standardize: Standardize column values (strip whitespace, normalize case)\n"
    "- trim: Trim whitespace from column values\n"
    "- correct_typo: Correct typos in categorical values\n\n"
    "Think step by step: examine the data quality report, identify the most "
    "impactful issue, then act."
)

In [ ]:
# === Helpers ===

def _extract_column(text):
    quoted = re.findall(r'["\']([\w]+)["\']', text)
    if quoted: return quoted[0]
    camel = re.findall(r'\b([A-Z][a-z]+(?:[A-Z][a-z]+)+)\b', text)
    if camel: return camel[0]
    return ""

def parse_action(text):
    m = re.search(r'\{[^{}]*"operation"[^{}]*\}', text, re.DOTALL)
    if m:
        try:
            data = json.loads(m.group())
            if "operation" in data and "column" in data: return data
        except json.JSONDecodeError: pass
    tl = text.lower()
    if "fill" in tl or "null" in tl:
        return {"operation": "fill_null", "column": _extract_column(text), "value": "median", "params": {}}
    if "type" in tl or "cast" in tl:
        return {"operation": "fix_type", "column": _extract_column(text), "value": "numeric", "params": {}}
    if "duplicate" in tl:
        return {"operation": "remove_duplicate", "column": "", "params": {}}
    if "standard" in tl:
        return {"operation": "standardize", "column": _extract_column(text), "params": {}}
    if "trim" in tl:
        return {"operation": "trim", "column": _extract_column(text), "params": {}}
    if "typo" in tl:
        return {"operation": "correct_typo", "column": _extract_column(text), "params": {}}
    return {"operation": "fill_null", "column": "", "value": "median", "params": {}}

def env_reset(seed=42):
    try:
        r = requests.post(f"{ENV_URL}/web/reset", json={"seed": seed}, timeout=30)
        r.raise_for_status()
        d = r.json()
        return d.get("observation", d)
    except Exception as e:
        return {"error": str(e)}

def env_step(action):
    try:
        r = requests.post(f"{ENV_URL}/web/step", json={"action": action}, timeout=30)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        return {"error": str(e), "reward": 0, "done": True, "observation": {}}

def format_obs(obs):
    return (
        f"Domain: {obs.get('domain', 'unknown')}\n"
        f"DQ Score: {obs.get('dq_score', 'N/A')}\n"
        f"DQ Report: {obs.get('dq_report', '')}\n"
        f"Columns: {obs.get('columns_info', '')}\n"
        f"Data Preview: {str(obs.get('data_preview', ''))[:500]}\n"
        f"Step {obs.get('step_number', '?')}/{obs.get('max_steps', '?')}\n\n"
        "Output a single JSON cleaning action."
    )

def generate(model, tokenizer, sys_prompt, user_prompt, max_new_tokens=256):
    msgs = [{"role": "system", "content": sys_prompt}, {"role": "user", "content": user_prompt}]
    ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, temperature=0.7, do_sample=True, pad_token_id=tokenizer.pad_token_id)
    resp = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    return re.sub(r'<think>.*?</think>', '', resp, flags=re.DOTALL).strip()

def run_episode(model, tokenizer, seed=42):
    obs = env_reset(seed)
    if "error" in obs: return {"error": obs["error"]}
    rewards, actions = [], []
    initial_dq = obs.get("dq_score", 0)
    for _ in range(MAX_STEPS):
        resp = generate(model, tokenizer, SYSTEM_PROMPT, format_obs(obs))
        action = parse_action(resp)
        actions.append(action)
        result = env_step(action)
        rewards.append(result.get("reward", 0))
        new_obs = result.get("observation", {})
        if isinstance(new_obs, dict): obs = {**obs, **new_obs}
        if result.get("done", False): break
    final_dq = obs.get("dq_score", initial_dq)
    return {
        "initial_dq": float(initial_dq), "final_dq": float(final_dq),
        "dq_improvement": float(final_dq - initial_dq),
        "rewards": [float(r) for r in rewards],
        "avg_reward": float(np.mean(rewards)) if rewards else 0,
        "steps": len(rewards), "actions": actions,
    }

# Verify env
try:
    r = requests.get(ENV_URL, timeout=10)
    print(f"Environment: {r.status_code} OK")
except Exception as e:
    print(f"Environment: UNREACHABLE ({e})")

In [ ]:
# === Load LoRA model ===
from unsloth import FastLanguageModel

print(f"Loading LoRA: {LORA_REPO}")
t0 = time.time()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_REPO, max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True, fast_inference=True, max_lora_rank=16, gpu_memory_utilization=0.6,
)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
FastLanguageModel.for_inference(model)
print(f"Loaded in {time.time()-t0:.1f}s")

In [ ]:
# === Run LoRA inference ===
print("=" * 60)
print("CLEANING — DataSage LoRA")
print("=" * 60)

lora_results = []
for di, domain in enumerate(DOMAINS):
    for ep in range(EPISODES_PER_DOMAIN):
        seed = di * 100 + ep + 1
        print(f"  [{domain}] ep {ep+1}/{EPISODES_PER_DOMAIN} seed={seed}", end=" ")
        t0 = time.time()
        result = run_episode(model, tokenizer, seed)
        result["domain"] = domain
        result["seed"] = seed
        result["latency_s"] = time.time() - t0
        lora_results.append(result)
        print(f"DQ={result.get('final_dq',0):.4f} R={result.get('avg_reward',0):.4f} steps={result.get('steps',0)} {result['latency_s']:.0f}s")

valid = [r for r in lora_results if "error" not in r]
if valid:
    print(f"\nLoRA summary ({len(valid)} eps): DQ={np.mean([r['final_dq'] for r in valid]):.4f} R={np.mean([r['avg_reward'] for r in valid]):.4f}")

# Free
del model, tokenizer
torch.cuda.empty_cache()
print("GPU memory freed.")

In [ ]:
# === Load base model (no LoRA) ===
print(f"\nLoading base model: {BASE_MODEL}")
t0 = time.time()
base_m, base_t = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True, fast_inference=True, max_lora_rank=16, gpu_memory_utilization=0.6,
)
if base_t.pad_token is None: base_t.pad_token = base_t.eos_token
FastLanguageModel.for_inference(base_m)
print(f"Loaded in {time.time()-t0:.1f}s")

In [ ]:
# === Run base model inference (1 ep per domain) ===
print("\nBASE MODEL — Cleaning")
base_results = []
for di, domain in enumerate(DOMAINS):
    seed = di * 100 + 1
    print(f"  [{domain}] seed={seed}", end=" ")
    result = run_episode(base_m, base_t, seed)
    result["domain"] = domain
    result["seed"] = seed
    base_results.append(result)
    print(f"DQ={result.get('final_dq',0):.4f} R={result.get('avg_reward',0):.4f}")

del base_m, base_t
torch.cuda.empty_cache()

In [ ]:
# === Save results ===
def summarize(results):
    valid = [r for r in results if "error" not in r]
    if not valid: return {"error": "no valid episodes"}
    return {
        "final_dq_mean": float(np.mean([r["final_dq"] for r in valid])),
        "final_dq_std": float(np.std([r["final_dq"] for r in valid])),
        "avg_reward_mean": float(np.mean([r["avg_reward"] for r in valid])),
        "dq_improvement_mean": float(np.mean([r["dq_improvement"] for r in valid])),
        "steps_mean": float(np.mean([r["steps"] for r in valid])),
        "n_episodes": len(valid),
        "per_domain": {
            d: {
                "final_dq_mean": float(np.mean([r["final_dq"] for r in valid if r.get("domain")==d])),
                "avg_reward_mean": float(np.mean([r["avg_reward"] for r in valid if r.get("domain")==d])),
                "n": len([r for r in valid if r.get("domain")==d]),
            } for d in DOMAINS if any(r.get("domain")==d for r in valid)
        },
    }

output = {
    "task": "cleaning",
    "datasage_lora": summarize(lora_results),
    "base_model": summarize(base_results),
    "raw_lora": [{k: v for k, v in r.items() if k != "actions"} for r in lora_results],
    "raw_base": [{k: v for k, v in r.items() if k != "actions"} for r in base_results],
    "metadata": {
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "lora_repo": LORA_REPO, "base_model": BASE_MODEL,
        "env_url": ENV_URL, "episodes_per_domain": EPISODES_PER_DOMAIN,
    },
}

with open("cleaning_results.json", "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved: cleaning_results.json ({os.path.getsize('cleaning_results.json')/1024:.1f} KB)")

if IN_COLAB:
    from google.colab import files
    files.download("cleaning_results.json")